#Prerequisites

##Load employee.csv

In [0]:
employee_df=spark.read.csv(path="/Volumes/quickstart_catalog/quickstart_schema/sandbox/dataset/employee.csv",header=True,inferSchema=True, sep="|", quote="'")
employee_df.display()

#Handling Missing Records

##Dropping Missing Records

###dropping row for which there is any null values present in the dataframe 

In [0]:
employee_df.na.drop(how="any").display()

###dropping row which has all null values

In [0]:
employee_df.na.drop(how="all").display()

### dropping null based on specific filter 

In [0]:
employee_df.na.drop(subset=["id", "name"]).display()

###ques- filter name 

In [0]:
from pyspark.sql.functions import col
#list of employees whose id is null
employee_df.filter(col("id").isNull()).select("name").display()

In [0]:
#drop records when both id and name is null using filter

from pyspark.sql.functions import col
employee_df.filter(~(col("id").isNull() & col("name").isNull())).display()

##Filling Missing Records

###Approach 1

In [0]:
employee_df.na.fill("NULL IN SOURCE", subset=["name", "gen"]).na.fill(
    -1, subset=["id"]
).na.fill(0, subset=["exp"]).display()

###Approach 2

In [0]:
MISSING_DEFAULT_VALUES={"id":-1,"name":"Anonymous","exp":0}
employee_df.na.fill(MISSING_DEFAULT_VALUES).display()

###Approach 3

In [0]:
from pyspark.sql.functions import when
employee_df.withColumn("name", when(col("name").isNull(), "Anonymous").otherwise(col("name"))).display()

In [0]:
from pyspark.sql.functions import when
employee_df.withColumn("id", when(col("id").isNull(), -1).otherwise(col("id"))).withColumn("exp", when(col("exp").isNull(), 0).otherwise(col("exp"))).display()

> The error is clear: na.fill() only accepts primitive types (bool, dict, float, int, or str) as the value argument, but the code is passing today() which returns a Column object, not a date value.

To fill date columns with the current date, I need to use withColumn() with when() instead of na.fill()

In [0]:
from pyspark.sql.functions import current_date as today
employee_df.na.fill(today(), subset=["doj"]).display()


In [0]:
from pyspark.sql.functions import current_date as today, when, col
employee_df.withColumn("doj", when(col("doj").isNull(), today()).otherwise(col("doj"))).display()

In [0]:
from pyspark.sql.functions import avg
avg_exp=employee_df.select(avg("exp")).collect()[0][0]
avg_exp

In [0]:
MISSING_DEFAULT_VALUES = {"id": -1, "name": "Anonymous", "exp": round(avg_exp)}
employee_df.na.fill(MISSING_DEFAULT_VALUES).display()

In [0]:
from pyspark.sql.functions import when
employee_df.withColumn("gen", when(col("gen")=="M", "Male").when(col("gen")=="F", "Female").when(col("gen")=="T", "Transgender").otherwise(col("gen"))).display()


In [0]:
from pyspark.sql.functions import avg
avg_exp=employee_df.select(avg("exp")).collect()[0][0]
mode_gender=(employee_df.groupBy("gen").count().sort(col("count"),ascending=False).first()[0][0])

MISSING_DEFAULT_VALUES = {"id": -1, "name": "Anonymous", "exp": round(avg_exp), "gen": mode_gender}
employee_df.na.fill(MISSING_DEFAULT_VALUES).display()

In [0]:
employee_df.na.replace(-1,1, subset=["exp"]).display()

In [0]:
employee_df.na.replace({"M": "Male", "F": "Female", "T": "Transgender"}, subset=["gen"]).display()